In [1]:
import os
import tempfile

from redislite.falkordb_client import (
    FalkorDB,  # embedded FalkorDB: pip install falkordblite
)

import opencyver
from opencyver import PropertiesValidator, SchemaValidator, SyntaxValidator

In [2]:
# Start an embedded FalkorDB (FalkorDBLite) and load a small SustainGraph. No server is needed.
# To use a FalkorDB server instead:  driver = falkordb.FalkorDB(host="localhost", port=6379)
SEED = """
CREATE (g1:Goal {code: '1', title: 'End poverty', description: 'End poverty in all its forms everywhere'}),
       (g2:Goal {code: '2', title: 'Zero hunger', description: 'End hunger, achieve food security and improved nutrition'}),
       (t1:Target {code: '1.1', title: 'Extreme poverty', description: 'Eradicate extreme poverty for all people everywhere'}),
       (t2:Target {code: '1.2', title: 'Poverty in all its dimensions', description: 'Reduce by at least half the proportion of people living in poverty'}),
       (t3:Target {code: '2.1', title: 'End hunger', description: 'End hunger and ensure access to safe, nutritious and sufficient food'}),
       (i1:Indicator {code: '1.1.1', description: 'Proportion of the population living below the international poverty line'}),
       (i2:Indicator {code: '1.2.1', description: 'Proportion of the population living below the national poverty line'}),
       (i3:Indicator {code: '2.1.1', description: 'Prevalence of undernourishment'}),
       (:GeoArea:Area {name: 'Greece', M49code: '300', ISOalpha2code: 'GR', ISOalpha3code: 'GRC', EUcode: 'EL'}),
       (:GeoArea:Area {name: 'Germany', M49code: '276', ISOalpha2code: 'DE', ISOalpha3code: 'DEU', EUcode: 'DE'}),
       (g1)-[:HAS_TARGET]->(t1), (g1)-[:HAS_TARGET]->(t2), (g2)-[:HAS_TARGET]->(t3),
       (t1)-[:HAS_INDICATOR]->(i1), (t2)-[:HAS_INDICATOR]->(i2), (t3)-[:HAS_INDICATOR]->(i3)
"""

driver = FalkorDB(os.path.join(tempfile.mkdtemp(), "sustaingraph.db"))
database_name = "sustaingraph"  # in FalkorDB the database name is the name of the graph
driver.select_graph(database_name).query(SEED)
print("Graph", repr(database_name), "loaded; OpenCyVer", opencyver.__version__)

Graph 'sustaingraph' loaded; OpenCyVer 3.0.1


Database demo : SustainGraph (a small sample is created in the notebook)

![Basic schema of the SustainGraph](images/basic_schema.png)

# Syntax Validator

### 1. Syntax Error

In [3]:
query = """
MATCH (g:GeoArea)
RETURN g
"""
syntax_validator = SyntaxValidator(driver)
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"Syntax Valid: {is_valid}")
print("Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

Syntax Valid: True
Syntax Error Metadata:


### 2. Parameter not provided


In [4]:
query = """
MATCH (g:GeoArea{code:$code_param})
RETURN COUNT(g)
"""
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"--->Syntax Valid: {is_valid}")
print("--->Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

--->Syntax Valid: False
--->Syntax Error Metadata:
OpenCyVer.Warning.ParameterNotProvided: 
 Did not supply query with enough parameters. Missing parameters: code_param


### 3. Unsupported Cypher
FalkorDB does not support some Cypher that other databases accept, such as `&` between relationship types (`[:HAS_TARGET&HAS]`), label negation (`:!Label`) and node label alternatives (`:A|B`). OpenCyVer reports them as syntax errors.

In [5]:
query = """
MATCH (g:Goal)-[:HAS_TARGET&HAS_INDICATOR]->(t:Target)
RETURN COUNT(g)
"""
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"--->Syntax Valid: {is_valid}")
print("--->Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

--->Syntax Valid: False
--->Syntax Error Metadata:
FalkorDB.ClientError.Statement.SyntaxError: 
 errMsg: Invalid input '&': expected '|', '*', '{', a parameter or ']' line: 2, column: 28, offset: 28 errCtx: MATCH (g:Goal)-[:HAS_TARGET&HAS_INDICATOR]->(t:Target) errCtxOffset: 27


### 4.Arithmetic Error (explicit division by zero)
Regex pattern to catch explicit " / 0", " / 0.0", or " / (0)"

In [6]:
query = """
MATCH (g:GeoArea)
RETURN COUNT(g)/0
"""
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"--->Syntax Valid: {is_valid}")
print("--->Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

--->Syntax Valid: False
--->Syntax Error Metadata:
OpenCyVer.Error.ArithmeticError: 
 Division by zero.


### 5.Conflicting Variables
check_multilabeled_nodes (bool): Whether to check for multilabeled nodes when identifying conflicting variables based on their labels(default: True).

In [7]:
query = """
MATCH (g:GeoArea)
MATCH (g:Goal)
RETURN *
"""
syntax_validator = SyntaxValidator(driver, check_multilabeled_nodes=True)
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"--->Syntax Valid: {is_valid}")
print("--->Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

--->Syntax Valid: False
--->Syntax Error Metadata:
ConflictingVariablesError: 
 Variable 'g' has conflicting labels: GeoArea, Goal


In [8]:
# GeoArea and Area are multilabels
query = """
MATCH (g:GeoArea)
MATCH (g:Area)
RETURN g
"""
print("-----------Consider multilables--------------")
syntax_validator = SyntaxValidator(driver, check_multilabeled_nodes=True)
is_valid, syntax_metadata = syntax_validator.validate(
    query, database_name=database_name
)
print(f"--->Syntax Valid: {is_valid}")
print("--->Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

-----------Consider multilables--------------
--->Syntax Valid: True
--->Syntax Error Metadata:


# Schema Validator

### 1.Path Pattern Inference feature

In [9]:
query = """
MATCH (g:Goal)-[:HAS_TARGET]->(t:Target)-[:HAS_INDICATOR]->(i:Indicator) 
RETURN *
"""

schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
# Validation
print("--" * 50)
schema_score, schema_metadata = schema_validator.validate(
    query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Goal)', '(:Indicator)', '(:Target)']
Extracted Relationships Patterns : ['()-[:HAS_INDICATOR]-()', '()-[:HAS_TARGET]-()']
Extracted Path patterns : ['(:Goal)-[:HAS_TARGET]->(:Target)', '(:Target)-[:HAS_INDICATOR]->(:Indicator)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 1.0
Schema Validation Metadata:


### 2.Unknown Label / Relationship Type

In [10]:
query = """
MATCH (g:Goals)-[:HAS_TARGET]->(t:Target)-[:HAS_INDICATORs]->(i:Indicator) 
RETURN *
"""

schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
print("--" * 50)
# Validation
schema_score, schema_metadata = schema_validator.validate(
    query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Goals)', '(:Indicator)', '(:Target)']
Extracted Relationships Patterns : ['()-[:HAS_INDICATORs]-()', '()-[:HAS_TARGET]-()']
Extracted Path patterns : ['(:Goals)-[:HAS_TARGET]->(:Target)', '(:Target)-[:HAS_INDICATORs]->(:Indicator)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 0.38888888888888884
Schema Validation Metadata:
OpenCyVer.Warning.UnknownLabel: 
 One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: Goals)
OpenCyVer.Warning.UnknownRelationshipType: 
 One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that it is available when you run this statement in your

#### If multiple options for the inferred paths are available then the first combination (alphabetically) is selected.

In [11]:
query = """
MATCH (g:Goal)-[]->(t)
RETURN *
"""

schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
# Validation
print("--" * 50)
schema_score, schema_metadata = schema_validator.validate(
    query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Goal)']
Extracted Relationships Patterns : []
Extracted Path patterns : ['(:Goal)-[:HAS_TARGET]->(:Target)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 1.0
Schema Validation Metadata:


### 3. Paths:
* Unknown path
* Wrong direction Path

In [12]:
# The labels Goal and Indicator exist, and so does HAS_INDICATOR, but Goal is not connected to Indicator directly
query = """
MATCH (g:Goal)-[:HAS_INDICATOR]->(i:Indicator)
RETURN *
"""

schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
# Validation
print("--" * 50)
schema_score, schema_metadata = schema_validator.validate(
    query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Goal)', '(:Indicator)']
Extracted Relationships Patterns : ['()-[:HAS_INDICATOR]-()']
Extracted Path patterns : ['(:Goal)-[:HAS_INDICATOR]->(:Indicator)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 0.6666666666666666
Schema Validation Metadata:
 The specified path (:Goal)-[:HAS_INDICATOR]->(:Indicator) does not exist in the knowledge graph schema. The path (g:Goal)-[:HAS_INDICATOR]->(i:Indicator) in the Cypher statement is wrong.


In [13]:
query = """
MATCH (g:Goal)<-[:HAS_TARGET]-(t:Target{code:1})
RETURN *
"""

schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
# Validation
print("--" * 50)
schema_score, schema_metadata = schema_validator.validate(
    query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Goal)', '(:Target)']
Extracted Relationships Patterns : ['()-[:HAS_TARGET]-()']
Extracted Path patterns : ['(:Goal)<-[:HAS_TARGET]-(:Target)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 0.6666666666666666
Schema Validation Metadata:
 The specified path (:Goal)<-[:HAS_TARGET]-(:Target) has the opposite direction in the knowledge graph schema. The path (g:Goal)<-[:HAS_TARGET]-(t:Target{code:1}) in the Cypher statement should be changed to  (g:Goal)-[:HAS_TARGET]->(t:Target{code:1}).


# Properties Validator

### 1. How extract - validate works
### 2. Unknown Property
### 3. Invalid Property Access By Label

In [14]:
# Evaluate accessed properties by defined labels and unknown variables (inferred labels)
query = """
MATCH (g:Goal{title:"End"})
WHERE g.code = '1' 
RETURN *
"""

props_validator = PropertiesValidator(driver)

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    query, strict=False, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, properties_metadata = props_validator.validate(
    query, database_name=database_name, strict=False
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in properties_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Accessed properties by variables: {}
Accessed properties by labels (including inferred): {'Goal': ['title', 'code']}
----------------------------------------------------------------------------------------------------
Properties Validation Score: 1.0
Properties Validation Metadata:


### 4. Inference labels considering properties

In [15]:
# Evaluate accessed properties by defined labels and unknown variables (inferred labels)
query = """
MATCH (g:Goal{title:"End"})-[:HAS_TARGET]->(t) 
WHERE g.code = '1' and t.code = '1.1'
RETURN *
"""

props_validator = PropertiesValidator(driver)

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    query, strict=False, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, properties_metadata = props_validator.validate(
    query, database_name=database_name, strict=False
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in properties_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Accessed properties by variables: {'t': ['code']}
Accessed properties by labels (including inferred): {'Goal': ['title', 'code'], 'Target': ['code']}
----------------------------------------------------------------------------------------------------
Properties Validation Score: 1.0
Properties Validation Metadata:


In [16]:
# Evaluate accessed properties by defined labels and unknown variables (inferred labels)
query = """
MATCH (g:Goal{title:"End"})-[:HAS_TARGET]->(t) 
WHERE g.code = '1' and t.code = '1.1'
MATCH (a) WHERE a.dateOfReport = 1 
RETURN a,t
"""

props_validator = PropertiesValidator(driver)

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    query, strict=False, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, properties_metadata = props_validator.validate(
    query, strict=False, database_name=database_name
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in properties_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Accessed properties by variables: {'t': ['code'], 'a': ['dateOfReport']}
Accessed properties by labels (including inferred): {'Goal': ['title', 'code', 'dateOfReport'], 'Target': ['code']}
----------------------------------------------------------------------------------------------------
Properties Validation Score: 0.75
Properties Validation Metadata:
OpenCyVer.Warning.UnknownPropertyKey: 
 One of the property names in your query is not available in the database, make sure you didn't misspell it or that it is available when you run this statement in your application (the missing property name is: dateOfReport)
 The label Goal does not have the following properties:  dateOfReport.


### 5. How parameter strict works in inference

In [17]:
# Evaluate accessed properties by defined labels and unknown variables (inferred labels)
query = """
MATCH (a) 
WHERE a.code = 1 and a.description = 'description_text' and a.EUcode ='EL' 
RETURN a
"""

props_validator = PropertiesValidator(driver)

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    query, strict=False, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, properties_metadata = props_validator.validate(
    query, database_name=database_name, strict=False
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in properties_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Accessed properties by variables: {'a': ['code', 'description', 'EUcode']}
Accessed properties by labels (including inferred): {'Goal': ['code', 'description', 'EUcode']}
----------------------------------------------------------------------------------------------------
Properties Validation Score: 0.6666666666666666
Properties Validation Metadata:
 The label Goal does not have the following properties:  EUcode.


In [18]:
# Evaluate accessed properties by defined labels and unknown variables (inferred labels)
query = """
MATCH (a) 
WHERE a.code = 1 and a.description = 'description_text' and a.EUcode ='EL' 
RETURN a
"""

props_validator = PropertiesValidator(driver)

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    query, strict=True, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, properties_metadata = props_validator.validate(
    query, strict=True, database_name=database_name
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in properties_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------


Accessed properties by variables: {'a': ['code', 'description', 'EUcode']}
Accessed properties by labels (including inferred): {}
----------------------------------------------------------------------------------------------------


Properties Validation Score: None
Properties Validation Metadata:
